# Row embeddings (ad hoc)

Chunk a saved spreadsheet's data rows, embed each chunk with Titan Text Embeddings v2, and save them as
`row_embeddings.json` in the document's `preprocessed/` folder. Once saved, the agent offers the
`semantic_row_search` tool whenever that spreadsheet is selected.

The document store comes from `.env` like the apps: the local document library by default, or S3 when
`DOCUMENTS_STORAGE=s3`. Re-uploading the spreadsheet removes its row embeddings; run this again afterwards.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # walk up to the repo root wherever the kernel started
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from backend.agents.embeddings import Embedder, embed_rows, row_chunks
from backend.retrieval.semantic import row_search_documents
from backend.storage.s3 import configured_store

FILENAME = "Location of 386 Pokémon in Gen 3.xlsx"  # a spreadsheet already saved in the document library
GROUP = None              # group folder in S3 mode, or None to use DOCUMENTS_GROUP_NAME
CHUNKING = "row"          # "row" or "row_group"
ROWS_PER_CHUNK = 10       # used by "row_group"

## 1. Load the saved spreadsheet

In [ ]:
store = configured_store()
if GROUP:
    store = store.for_group(GROUP)
doc = store.load(FILENAME)
{name: (sheet["sheet_type"], sheet["row_count"]) for name, sheet in doc.table_metadata["sheets"].items()}

## 2. Preview the chunks

Data sheets only; narrative sheets are already text blocks for `semantic_search`.

In [ ]:
chunks = row_chunks(doc, chunking=CHUNKING, rows_per_chunk=ROWS_PER_CHUNK)
print(f"{len(chunks)} chunks, longest {max(len(c['text']) for c in chunks):,} characters\n")
print(chunks[0]["text"])

## 3. Embed and save

One Titan call per chunk, 8 at a time.

In [ ]:
embedder = Embedder()
row_embeddings = embed_rows(doc, embedder, chunking=CHUNKING, rows_per_chunk=ROWS_PER_CHUNK)
print({k: v for k, v in row_embeddings["meta"].items() if k != "failed_chunks"})
if row_embeddings["meta"]["failed_chunks"]:
    print("Failed:", row_embeddings["meta"]["failed_chunks"][:5])
store.save_row_embeddings(doc.storage_ref["manifest_key"], row_embeddings)

## 4. Try the search the agent's tool runs

In [ ]:
doc = store.load(FILENAME)  # reload to read back what was saved
QUERY = "Where can I find Ralts?"  # change me
for match in row_search_documents({doc.id: doc}, QUERY, embedder, limit=5)["matches"]:
    print(f"--- {match['similarity']:.3f}  {match['sheet']} rows {match['source_rows']}\n{match['text'][:500]}\n")